In [23]:
import pandas as pd
import matplotlib as plt
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

cities = pd.read_csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2026/2026-05-12/cities.csv')
links = pd.read_csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2026/2026-05-12/links.csv')

In [4]:
cities.info()

<class 'pandas.DataFrame'>
RangeIndex: 5470 entries, 0 to 5469
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   id         5470 non-null   str    
 1   name       5470 non-null   str    
 2   lng        5470 non-null   float64
 3   lat        5470 non-null   float64
 4   country    5470 non-null   str    
 5   countrycd  5463 non-null   str    
 6   continent  5470 non-null   str    
dtypes: float64(2), str(5)
memory usage: 299.3 KB


In [5]:
links.info()

<class 'pandas.DataFrame'>
RangeIndex: 10596 entries, 0 to 10595
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   source  10596 non-null  str  
 1   target  10596 non-null  str  
dtypes: str(2)
memory usage: 165.7 KB


In [6]:
links.sample(10)

,source,target
6294,Q1134454,Q386991
9202,Q2292609,Q178855
9816,Q558530,Q520025
5251,Q34370,Q648
2930,Q234061,Q37333
35,Q766942,Q1023781
4017,Q916536,Q835810
3377,Q707030,Q851004
1142,Q4105,Q742160
9495,Q52896,Q16138


In [27]:
cities.sample(10)


,id,name,lng,lat,country,countrycd,continent
2637,Q1976493,Aghstafa,45.450000,41.116667,Azerbaijan,AZ,Asia
378,Q41262,Nottingham,-1.149167,52.955000,United Kingdom,GB,Europe
4925,Q602976,Klatovy,13.295089,49.395625,Czechia,CZ,Europe
3525,Q305697,Płoty,15.264722,53.808056,Poland,PL,Europe
360,Q170919,Pattaya,100.893004,12.949565,Thailand,TH,Asia
1193,Q572615,Beeskow,14.246585,52.172170,Germany,DE,Europe
4593,Q747388,Princeton,-81.095833,37.368056,United States of America,US,North America
2207,Q5332,Saratov,46.035000,51.530000,Russia,RU,Europe
2004,Q351427,Adama,39.266667,8.550000,Ethiopia,ET,Africa
4894,Q1018857,Nový Bydžov,15.490898,50.241553,Czechia,CZ,Europe


In [44]:
coords_df = cities[['id', 'lng', 'lat']]

links = links.merge(
    coords_df,
    left_on='source',
    right_on='id',
    how='left'
).drop(columns=['id'])

links = links.merge(
    coords_df,
    left_on='target',
    right_on='id',
    how='left',
    suffixes=('_source', '_target')
).drop(columns=['id'])

# Reduccion de 4 cruces a 2, la magia de pandas:
# Primera propuesta...

# links_with_lat_lng = links.merge(
#     cities[['id', 'lng']],
#     left_on='source',
#     right_on='id',
#     how='left'
# )

# links_with_lat_lng = links_with_lat_lng.merge(
#     cities[['id', 'lat']],
#     left_on='source',
#     right_on='id',
#     how='left'
# )
    
# links_with_lat_lng = links_with_lat_lng.drop(
#     columns=['id_x', 'id_y']
#     ).rename(
#     columns={"lng" : "lng_source", "lat" : "lat_source"}
# )

# links_with_lat_lng = links_with_lat_lng.merge(
#     cities[['id', 'lng']],
#     left_on='target',
#     right_on='id',
#     how='left',
# )

# links_with_lat_lng = links_with_lat_lng.merge(
#     cities[['id', 'lat']],
#     left_on='target',
#     right_on='id',
#     how='left'
# )

# links_with_lat_lng = links_with_lat_lng.drop(
#     columns=['id_x', 'id_y']
#     ).rename(
#     columns={"lng" : "lng_target", "lat" : "lat_target"}
# )

links.head(10)

,source,target,lng_source,lat_source,lng_target,lat_target
0,Q734532,Q873835,139.281500,36.197472,138.889917,36.259917
1,Q838366,Q242783,124.185250,24.344500,137.173028,34.954722
2,Q747362,Q271917,138.905444,35.551528,144.381722,42.984889
3,Q836836,Q385684,139.755444,36.178250,136.487500,35.979722
4,Q823988,Q386052,137.998389,34.768750,141.139139,39.144472
5,Q820496,Q386661,136.870667,35.332083,139.491111,35.338833
6,Q823489,Q428989,140.619889,38.002444,141.106583,42.412750
7,Q820496,Q495809,136.870667,35.332083,139.549167,35.856556
8,Q750587,Q496472,140.210917,37.126306,136.683667,35.062194
9,Q753864,Q502205,137.447750,36.871444,145.582861,43.330083


In [51]:
fig = go.Figure()

fig.add_trace(go.Scattergeo(
    lon = cities['lng'],
    lat = cities['lat'],
    hoverinfo = 'text',
    text = cities['name'] + ', ' + cities['country'],
    mode = 'markers',
    marker = dict(
        size = 2,
        color = 'rgb(255, 0, 0)',
        line = dict(
            width = 3,
            color = 'rgba(68, 68, 68, 0)'
        )
    )))

lons = np.empty(3 * len(links))
lons[::3] = links['lng_source']
lons[1::3] = links['lng_target']
lons[2::3] = np.nan

lats = np.empty(3 * len(links))
lats[::3] = links['lat_source']
lats[1::3] = links['lat_target']
lats[2::3] = np.nan

fig.add_trace(
    go.Scattergeo(
        lon = lons,
        lat = lats,
        mode = 'lines',
        line = dict(width = 0.5, color = 'red'),
        opacity = 0.5
    )
)

fig.update_layout(
    title_text = 'Sister Cities',
    showlegend = False,
    geo = go.layout.Geo(
        scope = 'world',
        projection_type = 'natural earth',
        showland = True,
        landcolor = 'rgb(243, 243, 243)',
        countrycolor = 'rgb(204, 204, 204)',
    ),
    height = 700,
)

fig.show()

In [26]:




# diccionario para buscar long y lat usando ID
coords = cities.set_index('id')[['lat', 'lng', 'name']].to_dict('index')
# df.index('id')[['lat, 'lng', 'name']].to_dict('index')

lons = []
lats = []

for _, row in links.iterrows():
    source = row['source']
    target = row['target']
    
    if source in coords and target in coords:
        # Añadir coordenadas de origen
        lons.append(coords[source]['lng'])
        lats.append(coords[source]['lat'])
        # Añadir coordenadas de destino
        lons.append(coords[target]['lng'])
        lats.append(coords[target]['lat'])
        # Añadir "None" para decirle a Plotly que levante el lápiz
        lons.append(None)
        lats.append(None)

# 3. CREAMOS LA FIGURA
fig = go.Figure()

# 4. AÑADIMOS LAS LÍNEAS EN UNA SOLA CAPA MASIVA
fig.add_trace(
    go.Scattergeo(
        lon=lons,
        lat=lats,
        mode='lines',
        line=dict(width=0.3, color='rgba(100, 150, 250, 0.4)'), # Líneas más finas y transparentes
        hoverinfo='none'
    )
)

# 5. AÑADIMOS LOS NODOS (Tu código original)
fig.add_trace(go.Scattergeo(
    lon = cities["lng"],
    lat = cities["lat"],
    hoverinfo = "text",
    text = cities['name'] + ", " + cities['country'],
    mode = 'markers',
    marker = dict(
        size = 1, # Aumenté un poco el tamaño a 3 para que destaquen más
        color = 'rgb(255, 100, 105)',
        line = dict(width=0) # Quitar el borde mejora aún más el rendimiento
    )
))

# 6. CONFIGURACIÓN DEL MAPA
fig.update_layout(
    title_text='Red Global de Ciudades (Optimizada)',
    showlegend=False,
    geo=dict(
        showland=True,
        landcolor='rgb(243, 243, 243)',
        countrycolor='rgb(204, 204, 204)',
    )
)

pio.renderers.default = 'iframe'
fig.show()

